# 1ª entrega — Análise Exploratória de Dados (AED) comentada
**Aluno(a):** _Gabriela Marino Koerich_ · **Disciplina:** Python Machine Learning

**Pergunta do projeto:** a partir de biomarcadores, é possível (1) classificar o paciente em `BCG`, `cancer`, `imunodeficiencia` ou `leucemia`, (2) prever o valor de `IFN-γ` e (3) encontrar grupos naturais (clusters)?

**Critério de sucesso:** F1-macro ≥ 0,75 na classificação; R² ≥ 0,70 na regressão.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../dados/biomarcadores.csv")
df.shape

(1000, 38)

## 1. Visão geral

In [4]:
df.head()

,CD34,TdT,MPO,CD19_1,CD20_1,CD22,CD3_1,CD7,BCR-ABL,PML-RARA,...,IgA_2,IgM,CD16,CD56,DHR,CH50,C3,C4,FOXP3,classe
0,0.119362,True,True,0.471228,0.398837,0.233216,0.329756,0.417118,True,True,...,Alto,Alto,247.894093,396.067195,0.967908,93,230,42,Normal,leucemia
1,0.112420,True,True,0.292042,0.370941,0.253135,0.340733,0.350616,True,True,...,Alto,Alto,213.416161,361.036864,0.972741,135,351,64,Normal,leucemia
2,0.175636,True,True,0.372872,0.257518,0.451758,0.487803,0.399385,True,True,...,Alto,Alto,296.635039,244.867766,0.971384,69,238,44,Normal,leucemia
3,0.078518,True,True,0.469943,0.495637,0.351360,0.332752,0.487439,True,True,...,Alto,Alto,564.401375,530.296010,0.962450,90,262,41,Normal,leucemia
4,0.061083,True,True,0.349343,0.381742,0.220939,0.479723,0.453555,True,True,...,Alto,Alto,238.791009,152.442218,0.955740,76,209,46,Normal,leucemia


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 38 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   CD34      1000 non-null   float64
 1   TdT       1000 non-null   bool   
 2   MPO       1000 non-null   bool   
 3   CD19_1    1000 non-null   float64
 4   CD20_1    1000 non-null   float64
 5   CD22      1000 non-null   float64
 6   CD3_1     964 non-null    float64
 7   CD7       1000 non-null   float64
 8   BCR-ABL   1000 non-null   bool   
 9   PML-RARA  1000 non-null   bool   
 10  NMP22     1000 non-null   int64  
 11  BTA       961 non-null    object 
 12  hTERT     954 non-null    object 
 13  uCyt+     963 non-null    object 
 14  FGFR3     945 non-null    object 
 15  TP53      1000 non-null   float64
 16  PPD       904 non-null    float64
 17  IFN-γ     1000 non-null   float64
 18  IL-2      1000 non-null   object 
 19  TNF-α     1000 non-null   object 
 20  Th1       1000 non-null   int64

In [8]:
# Classificação das variáveis por tipo
print(f"Observações: {df.shape[0]} | Variáveis: {df.shape[1]}")
print("\nTipos como o pandas leu:")
print(df.dtypes.value_counts())

# Lógicas = só True/False (ignorando faltantes), mesmo que lidas como object
logicas = [c for c in df.columns if set(df[c].dropna().unique()) <= {True, False}]
numericas = [c for c in df.select_dtypes(include="number").columns if c not in logicas]
categoricas = [c for c in df.columns if c not in logicas + numericas]

lidas_como_object = [c for c in logicas if df[c].dtype == "object"]
print("\nLógicas lidas como 'object' por terem faltantes:", lidas_como_object)

pd.DataFrame(
    {
        "Quantidade": [len(numericas), len(logicas), len(categoricas)],
        "Variáveis": [", ".join(numericas), ", ".join(logicas), ", ".join(categoricas)],
    },
    index=["Numéricas", "Lógicas", "Categóricas"],
)

Observações: 1000 | Variáveis: 38

Tipos como o pandas leu:
float64    17
object     12
int64       5
bool        4
Name: count, dtype: int64

Lógicas lidas como 'object' por terem faltantes: ['BTA', 'hTERT', 'uCyt+']


,Quantidade,Variáveis
Numéricas,22,"CD34, CD19_1, CD20_1, CD22, CD3_1, CD7, NMP22,..."
Lógicas,7,"TdT, MPO, BCR-ABL, PML-RARA, BTA, hTERT, uCyt+"
Categóricas,9,"FGFR3, IL-2, TNF-α, IgG_1, IgA_1, IgA_2, IgM, ..."


✍️ **Comentário 1:** O banco possui 1000 observações e 38 variáveis: 22 numéricas (17 contínuas e 5 inteiras), 7 lógicas e 9 categóricas, sendo classe a variável-alvo, com 4 categorias (BCG, cancer, imunodeficiencia e leucemia). O df.info() identifica apenas 4 variáveis do tipo bool, pois BTA, hTERT e uCyt+ contêm valores faltantes e foram lidas como object; conceitualmente, porém, também são lógicas. As categóricas são, em sua maioria, binárias (alto/baixo ou normal/alterado), o que facilita sua transformação em variáveis dummies para os modelos.

## 2. Valores faltantes e constantes

In [9]:
# Variáveis com faltantes: quantidade e proporção
faltantes = df.isna().sum()
faltantes = faltantes[faltantes > 0].sort_values(ascending=False)

total = df.isna().sum().sum()
linhas = df.isna().any(axis=1).sum()
print(f"Total de células faltantes: {total} ({total / df.size * 100:.1f}% do banco)")
print(f"Pacientes com pelo menos um faltante: {linhas} ({linhas / len(df) * 100:.0f}%)")

pd.DataFrame(
    {"Faltantes": faltantes, "Proporção (%)": (faltantes / len(df) * 100).round(1)}
)

Total de células faltantes: 309 (0.8% do banco)
Pacientes com pelo menos um faltante: 260 (26%)


,Faltantes,Proporção (%)
PPD,96,9.6
FGFR3,55,5.5
hTERT,46,4.6
BTA,39,3.9
uCyt+,37,3.7
CD3_1,36,3.6


In [10]:
# Variáveis constantes: um único valor em todos os pacientes (ignorando faltantes)
constantes = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]

restantes = [c for c in faltantes.index if c not in constantes]
print("Faltantes que ainda precisam de tratamento após remover as constantes:", restantes)

pd.DataFrame(
    {
        "Único valor": [df[c].dropna().unique()[0] for c in constantes],
        "Pacientes com esse valor": [df[c].notna().sum() for c in constantes],
        "Faltantes": [df[c].isna().sum() for c in constantes],
    },
    index=constantes,
)

Faltantes que ainda precisam de tratamento após remover as constantes: ['PPD', 'CD3_1']


,Único valor,Pacientes com esse valor,Faltantes
BTA,False,961,39
hTERT,False,954,46
uCyt+,False,963,37
FGFR3,Normal,945,55


✍️ **Comentário 2:** Seis variáveis apresentam valores faltantes, somando 309 células (0,8% do banco). A mais afetada é PPD, com 96 faltantes, seguida de FGFR3 (55), hTERT (46), BTA (39), uCyt+ (37) e CD3_1 (36). BTA, hTERT, uCyt+ e FGFR3 são constantes: todos os pacientes com valor registrado têm o mesmo resultado (False ou Normal). Variáveis constantes não carregam informação capaz de diferenciar as classes; além disso, aumentam a dimensionalidade e podem causar problemas em etapas como a padronização (desvio padrão zero). Por isso foram removidas. Com essa remoção, restam faltantes apenas em PPD e CD3_1. Em PPD, por ser muito assimétrica, os faltantes foram preenchidos com a média após a transformação logarítmica (PPD_log); nas demais numéricas, usou-se a mediana. Preencher em vez de excluir evita perder 260 pacientes (26% do banco), o que reduziria muito o tamanho da amostra.

## 3. Variável alvo — balanceamento das classes

In [ ]:
df["classe"].value_counts(normalize=True).round(3)

In [ ]:
sns.countplot(data=df, x="classe")
plt.title("Distribuição das classes")
plt.show()

✍️ **Comentário:** as classes estão balanceadas? Seria necessário SMOTE?

## 4. Medidas resumo e distribuições

In [ ]:
df.describe().T

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["PPD"], ax=ax[0]).set_title("PPD (original)")
sns.histplot(np.log1p(df["PPD"]), ax=ax[1]).set_title("log(1 + PPD)")
plt.show()

✍️ **Comentário:** a PPD é assimétrica? O que a transformação log mudou?

## 5. Variáveis × classe

In [ ]:
for var in ["TP53", "IFN-γ", "CD3_2", "CD8", "CD7"]:
    sns.boxplot(data=df, x="classe", y=var)
    plt.title(var)
    plt.show()

In [ ]:
pd.crosstab(df["FOXP3"], df["classe"], normalize="columns").round(2)

✍️ **Comentário:** quais biomarcadores separam bem as classes? Qual classe se destaca em IFN-γ?

## 6. Correlação

In [ ]:
corr = df.select_dtypes("number").corr(method="spearman")
plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap="coolwarm", center=0)
plt.show()

In [ ]:
# Pares muito correlacionados (|r| > 0,7) -> manter só um de cada par
pares = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
pares[pares.abs() > 0.7].sort_values()

✍️ **Comentário:** CD34, CD19_1, CD20_1, CD22, CD3_1 e CD7 formam um bloco correlacionado. Por que faz sentido manter só uma delas (aqui, CD7)?

## 7. Relatório automático (ydata-profiling)

In [ ]:
from ydata_profiling import ProfileReport

perfil = ProfileReport(df, title="AED - Biomarcadores", explorative=True)
perfil.to_file("relatorio_aed.html")
# perfil.to_notebook_iframe()  # para ver dentro do notebook

## 8. Conclusões da AED
✍️ Resuma em 4–6 tópicos:
- Qualidade dos dados (faltantes, constantes, duplicados) e como tratou;
- Balanceamento das classes;
- Transformações feitas (ex.: `PPD_log`);
- Variáveis selecionadas para os modelos: `TP53, IFN-γ, CD3_2, CD8, PPD_log, CD7, FOXP3`;
- Hipóteses para a modelagem.